In [ ]:
import json
from datetime import datetime as dt
from pathlib import Path

import polars as pl
import yaml
from dateutil.relativedelta import relativedelta

from clean import clean_os_version
from common import daterange, read_day

with open("../config.budapest.yaml", "r") as fp:
    config = yaml.safe_load(fp)

days = []
for i in config["days"]:
    days += list(daterange(i["start"], i["end"]))


In [2]:
def collect_os_stats(days: list) -> pl.DataFrame:
    os_stats = pl.DataFrame()
    for i in days:
        data = (
            read_day(i, config["dirs"]["raw_source"])
            .with_columns(
                pl.concat_str(
                    [pl.col("device_aid_hashed"), pl.col("device_aid_type")],
                    separator=".",
                ).alias("id_with_type"),
                pl.from_epoch(pl.col("timestamp"), time_unit="s")
                .dt.convert_time_zone(time_zone="Europe/Budapest")
                .dt.date()
                .alias("day"),
            )
            .rename({"device_aid_hashed": "device_id"})
        )
        data = data.select(
            pl.col.device_id,
            pl.col.id_with_type,
            pl.col.OS,
            pl.col.OS_version,
            pl.col.day,
        ).unique()
        data = data.with_columns(
            pl.col.OS_version.str.extract(r"(\d+)\..*", 1).cast(pl.Int16),
            pl.col("id_with_type").replace_strict(lookup),
        )
        os_stats = pl.concat([os_stats, data])
    return os_stats

In [3]:
OUTPUT = f"{config['dirs']['output']}/{config['city']}"

In [4]:
Path(f"{OUTPUT}/descriptives").mkdir(parents=True, exist_ok=True)

In [5]:
with open(f"{OUTPUT}/id_lookup/id_lookup.json", "r") as fp:
    lookup = json.load(fp)

In [6]:
try:
    os_stats = pl.read_parquet(f"{OUTPUT}/descriptives/os_stats.parquet")
except FileNotFoundError:
    # runs 50 m
    os_stats = collect_os_stats()
    os_stats.columns = ["id_with_type", "device_id", "OS", "OS_version", "day"]
    os_stats = os_stats.sort(["device_id", "day"])
    os_stats.write_parquet(f"{OUTPUT}/descriptives/os_stats.parquet")

In [7]:
os_type_count = os_stats.group_by(["device_id", "OS"]).agg(
    pl.col.id_with_type.unique().count().alias("count")
)

In [8]:
os_type_count["count"].sum() == len(os_type_count)

True

In [9]:
base = (
    os_stats.drop("id_with_type")
    .sort(
        ["device_id", "OS", "OS_version", "day"], descending=[False, False, False, True]
    )
    .unique(["device_id", "OS"], maintain_order=True, keep="first")
    .filter(pl.col.OS_version.is_not_null())
)

In [ ]:
def relative_age(
    year: int, month: int, ref_year: int = 2025, ref_month: int = 12
) -> int:
    """
    >>> relative_age(2017, 4, 2017, 4)
    0
    >>> relative_age(2017, 3, 2017, 4)
    1
    >>> relative_age(2016, 4, 2017, 4)
    12
    """
    years = (
        relativedelta(dt.date(ref_year, ref_month, 1), dt.date(year, month, 1)).years
        * 12
    )
    months = relativedelta(
        dt.date(ref_year, ref_month, 1), dt.date(year, month, 1)
    ).months
    return years + months


def relative_age_from_date(day: dt.date, ref: dt.date) -> int:
    return relative_age(day.year, day.month, ref_year=ref.year, ref_month=ref.month)


def get_release_date(os_versions: dict, os: str, os_version: int) -> dt.date:
    return os_versions[os.lower()][os_version]


def get_release_date_e(os_versions: dict, os: str, os_version: int) -> dt.date:
    return os_versions[os.str.to_lowercase()][os_version]


In [27]:
with open("../data/os_version.yaml", "r") as fp:
    os_versions = yaml.safe_load(fp)

In [28]:
get_release_date(os_versions, "Android", 5)

datetime.date(2014, 11, 4)

In [56]:
# base.with_columns(
#     get_release_date(os_versions, pl.col.OS, pl.col.OS_version).alias("release_date")
# )

In [ ]:
os_lookup = pl.DataFrame(
    [
        {"OS": os_name, "OS_version": version, "release_date": date}
        for os_name, versions in os_versions.items()
        for version, date in versions.items()
    ]
).with_columns(pl.col.OS.replace({"ios": "iOS", "android": "Android"}))

In [ ]:
enriched = (
    base.join(os_lookup, on=["OS", "OS_version"], how="left")
    .with_columns(
        (pl.col.day - pl.col.release_date).dt.total_days().alias("relative_age")
    )
    .rename({"day": "last_active_day"})
)

In [79]:
enriched.write_csv(f"{OUTPUT}/descriptives/phone_relative_age.csv")